# Open-Source Models — Variant Benchmark Generation

This notebook runs open-source language models on the Turkish reasoning
variant benchmark.

The historical open-source variant experiment included:

- Qwen 2.5 7B Instruct
- Llama 3.1 8B Instruct
- Gemma 2 9B IT (optional)

The variant experiment uses stricter JSON formatting instructions and a more
robust response parser than the original open-source base experiment.

For each variant, the notebook stores:

- variant identifier,
- corresponding base-question identifier,
- variant metadata,
- model reasoning,
- final answer,
- confidence,
- complete raw model response,
- generation errors, if any.

This notebook handles **generation only**.

Evaluation and base–variant comparison are performed separately.

## Important safety and reproducibility notes

Model execution is disabled by default.

No model is downloaded or run unless:

```python
RUN_GENERATION = True

## 1. Install dependencies

Open-source generation uses:

- PyTorch
- Hugging Face Transformers
- Accelerate
- bitsandbytes
- SentencePiece
- Hugging Face Hub

Evaluation libraries are intentionally excluded because evaluation is handled
in a separate notebook.

In [64]:
%pip install -q transformers accelerate bitsandbytes sentencepiece huggingface_hub pandas tqdm

In [65]:
from pathlib import Path
from datetime import datetime, timezone

import gc
import hashlib
import json
import os
import re
import sys

import pandas as pd
from tqdm.auto import tqdm


print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)

Python: 3.13.15
pandas: 2.2.3


## 2. Locate the project

The notebook supports:

1. a cloned GitHub repository,
2. a project copy stored in Google Drive.

Google Drive is particularly useful for long Colab runs because checkpoints
remain available if the runtime disconnects.

In [66]:
# ============================================================
# PROJECT CONFIGURATION
# ============================================================

USE_GOOGLE_DRIVE = True

DRIVE_PROJECT_PATH = Path(
    "/content/drive/MyDrive/turkish-llm-reasoning"
)


if USE_GOOGLE_DRIVE:

    try:
        from google.colab import drive

        drive.mount("/content/drive")

        PROJECT_ROOT = DRIVE_PROJECT_PATH

    except ImportError:

        raise RuntimeError(
            "Google Drive mounting is only available in Google Colab."
        )


else:

    current_path = Path.cwd().resolve()

    candidates = [
        current_path,
        *current_path.parents,
    ]

    PROJECT_ROOT = None


    for candidate in candidates:

        if (
            (candidate / "data").exists()
            and
            (candidate / "notebooks").exists()
        ):

            PROJECT_ROOT = candidate
            break


    if PROJECT_ROOT is None:

        raise FileNotFoundError(
            "Project root could not be located. "
            "Run the notebook from inside the repository "
            "or enable Google Drive mode."
        )


print("Project root:")
print(PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root:
/content/drive/MyDrive/turkish-llm-reasoning


## 3. Variant dataset configuration and reproducibility

Two variant-dataset profiles are supported:

- **original** — the snapshot used in the historical experiment.
- **current** — an optional evolving benchmark for future experiments.

The historical profile performs strict dataset-size checks.

The current profile does not assume a fixed number of variants.

A SHA-256 hash identifies the exact dataset file used by each generation run.

In [67]:
# ============================================================
# DATASET CONFIGURATION
# ============================================================

DATASET_PROFILE = "smoke"


DATASET_PROFILES = {

    "original": {
        "path": (
            PROJECT_ROOT
            / "data"
            / "variants"
            / "variants.csv"
        ),

        "expected_raw_rows": 257,
        "expected_usable_rows": 257,

        "description": (
            "Original open-source variant experiment snapshot"
        ),
    },


    "current": {
        "path": (
            PROJECT_ROOT
            / "data"
            / "variants"
            / "variants_current.csv"
        ),

        "expected_raw_rows": None,
        "expected_usable_rows": None,

        "description": (
            "Current evolving variant benchmark"
        ),
    },

    "smoke": {
    "path": (
        PROJECT_ROOT
        / "data"
        / "variants"
        / "smoke_variants_v1.csv"
    ),

    "expected_raw_rows": 15,
    "expected_usable_rows": 15,

    "description": (
        "Small smoke-test variant dataset for pipeline validation"
    ),
    },
}


if DATASET_PROFILE not in DATASET_PROFILES:

    raise ValueError(
        f"Unknown dataset profile: {DATASET_PROFILE}"
    )


DATASET_CONFIG = DATASET_PROFILES[
    DATASET_PROFILE
]

VARIANT_DATA_PATH = DATASET_CONFIG[
    "path"
]


print("Dataset profile:")
print(f"  Name:        {DATASET_PROFILE}")
print(f"  Description: {DATASET_CONFIG['description']}")
print(f"  Path:        {VARIANT_DATA_PATH}")

Dataset profile:
  Name:        smoke
  Description: Small smoke-test variant dataset for pipeline validation
  Path:        /content/drive/MyDrive/turkish-llm-reasoning/data/variants/smoke_variants_v1.csv


In [68]:
def calculate_file_sha256(filepath):

    sha256 = hashlib.sha256()

    with open(filepath, "rb") as file:

        for chunk in iter(
            lambda: file.read(8192),
            b"",
        ):

            sha256.update(chunk)

    return sha256.hexdigest()

In [69]:
if not VARIANT_DATA_PATH.exists():

    raise FileNotFoundError(
        f"Variant dataset not found: {VARIANT_DATA_PATH}"
    )


raw_variants_df = pd.read_csv(
    VARIANT_DATA_PATH
)


REQUIRED_FIELDS = [
    "variant_id",
    "base_question_id",
    "category",
    "difficulty",
    "variant_type",
    "morphological_feature",
    "context",
    "question",
    "gold_evidence",
    "gold_answer",
]


missing_columns = [
    column
    for column in REQUIRED_FIELDS
    if column not in raw_variants_df.columns
]


if missing_columns:

    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


variants_df = (
    raw_variants_df
    .dropna(
        subset=REQUIRED_FIELDS
    )
    .copy()
)


DATASET_SHA256 = calculate_file_sha256(
    VARIANT_DATA_PATH
)


print(
    f"Raw variant dataset:    "
    f"{len(raw_variants_df)} rows"
)

print(
    f"Usable variant dataset: "
    f"{len(variants_df)} rows"
)

print(
    f"Excluded:               "
    f"{len(raw_variants_df) - len(variants_df)} rows"
)

print(
    f"Unique base questions:  "
    f"{variants_df['base_question_id'].nunique()}"
)

print(
    f"Variant types:          "
    f"{variants_df['variant_type'].nunique()}"
)

print(
    f"Dataset SHA-256:        "
    f"{DATASET_SHA256}"
)

Raw variant dataset:    15 rows
Usable variant dataset: 15 rows
Excluded:               0 rows
Unique base questions:  5
Variant types:          8
Dataset SHA-256:        08f38dc18431f08a3574af59042aefaca717015c39133a254b21a551934a2443


In [70]:
expected_raw_rows = DATASET_CONFIG[
    "expected_raw_rows"
]

expected_usable_rows = DATASET_CONFIG[
    "expected_usable_rows"
]


if (
    expected_raw_rows is not None
    and len(raw_variants_df) != expected_raw_rows
):

    raise ValueError(
        f"Dataset profile '{DATASET_PROFILE}' expects "
        f"{expected_raw_rows} raw rows, "
        f"but {len(raw_variants_df)} were found."
    )


if (
    expected_usable_rows is not None
    and len(variants_df) != expected_usable_rows
):

    raise ValueError(
        f"Dataset profile '{DATASET_PROFILE}' expects "
        f"{expected_usable_rows} usable rows, "
        f"but {len(variants_df)} were found."
    )


if not raw_variants_df["variant_id"].is_unique:

    raise ValueError(
        "variant_id values must be unique."
    )


print("✓ Variant dataset validation passed.")

✓ Variant dataset validation passed.


### Historical dataset note

The canonical snapshot used by this repository contains **257 variants**
derived from **85 base questions** across **11 variant types**.

These values describe the historical dataset snapshot only.

Future versions may contain additional questions, linguistic transformations,
or variant types.

## 4. Generation configuration and model registry

The historical open-source variant experiment used:

- Qwen 2.5 7B Instruct,
- Llama 3.1 8B Instruct,
- Gemma 2 9B IT as an optional additional model.

Unlike the earlier open-source base experiment, the variant generation pipeline
used `max_new_tokens=1024`.

The longer generation limit was introduced to reduce truncated structured
responses.

New Hugging Face causal language models can be added by creating another entry
in `MODEL_REGISTRY`.

In [71]:
# ============================================================
# GENERATION CONFIGURATION
# ============================================================

RUN_GENERATION = True

CONFIRM_MULTIPLE_MODELS = False

RUN_LABEL = "smoke_v1"

CHECKPOINT_EVERY = 5


MODEL_REGISTRY = {

    "qwen2.5-7b-instruct": {
        "model_id": "Qwen/Qwen2.5-7B-Instruct",
        "display_name": "Qwen 2.5 7B Instruct",
        "historical": True,
        "max_new_tokens": 1024,
    },


    "llama-3.1-8b-instruct": {
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "display_name": "Llama 3.1 8B Instruct",
        "historical": True,
        "max_new_tokens": 1024,
    },


    "gemma-2-9b-it": {
        "model_id": "google/gemma-2-9b-it",
        "display_name": "Gemma 2 9B IT",
        "historical": True,
        "max_new_tokens": 1024,
    },
}


ORIGINAL_OPEN_VARIANT_MODELS = [
    "qwen2.5-7b-instruct",
    "llama-3.1-8b-instruct",
    "gemma-2-9b-it",
]


# Safe default: one model.
SELECTED_MODELS = [
    "qwen2.5-7b-instruct",
]

In [72]:
if not SELECTED_MODELS:

    raise ValueError(
        "At least one model must be selected."
    )


unknown_models = [
    model
    for model in SELECTED_MODELS
    if model not in MODEL_REGISTRY
]


if unknown_models:

    raise ValueError(
        f"Unknown model(s): {unknown_models}"
    )


if len(SELECTED_MODELS) != len(
    set(SELECTED_MODELS)
):

    raise ValueError(
        "SELECTED_MODELS contains duplicate entries."
    )


print("Selected models:")

for model_label in SELECTED_MODELS:

    config = MODEL_REGISTRY[
        model_label
    ]

    print(
        f"  - {config['display_name']} "
        f"({config['model_id']})"
    )

Selected models:
  - Qwen 2.5 7B Instruct (Qwen/Qwen2.5-7B-Instruct)


## 5. GPU and Hugging Face access

Generation requires a CUDA-capable GPU.

A Hugging Face token can be provided through:

- the `HF_TOKEN` environment variable,
- Google Colab Secrets.

The token is never stored in experiment outputs.

Some model repositories may additionally require the user to accept their
license or access conditions on Hugging Face.

In [73]:
def get_secret(name):

    value = os.getenv(name)

    if value:
        return value


    try:

        from google.colab import userdata

        return userdata.get(name)

    except Exception:

        return None

In [74]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)


if RUN_GENERATION:

    if not torch.cuda.is_available():

        raise RuntimeError(
            "A CUDA-capable GPU is required for generation."
        )


    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            1,
        ),
        "GB",
    )


else:

    print(
        "Generation is disabled. "
        "GPU availability is not required yet."
    )

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 95.0 GB


In [75]:
def login_to_huggingface_if_available():

    token = get_secret(
        "HF_TOKEN"
    )


    if not token:

        print(
            "HF_TOKEN not found. "
            "Continuing without authenticated login."
        )

        return


    from huggingface_hub import login

    login(
        token=token,
        add_to_git_credential=False,
    )

    print(
        "✓ Hugging Face authentication completed."
    )

## 6. Variant generation prompt

The variant experiment uses the same reasoning task as the other generation
notebooks but adds stricter formatting instructions.

These additional rules were introduced to reduce malformed JSON responses,
especially responses wrapped in Markdown or containing invalid line breaks.

The linguistic variant itself remains the experimental input change.

In [76]:
BASE_SYSTEM_PROMPT = """Sen bir problem çözme asistanısın. Sana verilen bağlam (context) ve soruyu (question) analiz edeceksin.

GÖREV:
1. Bağlamdaki bilgileri kullanarak adım adım düşün.
2. Her adımda hangi bilgiyi kullandığını açıkça belirt.
3. Sonuca ulaş ve cevabını ver.

ÇIKTI FORMATI (JSON):
{
    "reasoning": "Adım adım düşünce sürecin. Her adımda bağlamdan hangi bilgiyi kullandığını yaz.",
    "answer": "Nihai cevabın (kısa ve net)",
    "confidence": 0-100 arası bir sayı (cevabından ne kadar eminsin)
}

ÖNEMLİ:
- Sadece bağlamda verilen bilgileri kullan, dışarıdan bilgi ekleme.
- Cevabın SADECE geçerli bir JSON objesi olmalı, başka bir şey yazma.
- Türkçe yanıt ver."""

In [77]:
OPEN_SOURCE_SYSTEM_PROMPT = BASE_SYSTEM_PROMPT + """

ÇOK ÖNEMLİ FORMAT KURALI:
- Cevabını YALNIZCA tek bir JSON nesnesi olarak ver.
- JSON'u markdown kod bloğu (```json) içinde verme.
- JSON'dan önce veya sonra HİÇBİR açıklama, selamlama, özet ekleme.
- 'reasoning' alanını TEK bir string olarak yaz, ayrı string listesi olarak değil.
- 'reasoning' içinde satır geçişi kullanmak istiyorsan \\n yaz, literal newline değil.
- Cevabını JSON'un kapanış süslü parantezi ile bitir.
"""

In [78]:
def build_user_prompt(
    context,
    question,
):

    return f"""BAĞLAM:
{context}

SORU:
{question}

Yukarıdaki kurallara uygun, SADECE geçerli JSON formatında ve markdown sarmadan cevap ver."""

In [79]:
PROMPT_REFERENCE = (
    OPEN_SOURCE_SYSTEM_PROMPT
    + "\n---USER-PROMPT---\n"
    + build_user_prompt(
        "{context}",
        "{question}",
    )
)


PROMPT_SHA256 = hashlib.sha256(
    PROMPT_REFERENCE.encode("utf-8")
).hexdigest()


print("Prompt SHA-256:")
print(PROMPT_SHA256)

Prompt SHA-256:
fdc7227f4bf1480e69c921be7dbcb934c81ebe985942ae72d6b3f84256593279


## 7. Robust JSON response parser

The historical variant experiment introduced a more defensive parser because
some locally generated responses were structurally close to JSON but still
invalid.

The parser first attempts normal JSON extraction.

If that fails, four recovery strategies are attempted:

1. Markdown-wrapper recovery,
2. literal newline / tab repair,
3. concatenated reasoning-string recovery,
4. regular-expression recovery.

The complete raw response is preserved regardless of parsing success.

In [80]:
def basic_extract_json(raw):
    """
    Basic JSON extraction before variant-specific fallbacks.
    """

    if not raw or not isinstance(raw, str):

        return None


    raw = raw.strip()


    # --------------------------------------------------------
    # 1. Direct JSON
    # --------------------------------------------------------

    try:

        parsed = json.loads(raw)

        if isinstance(parsed, dict):

            return parsed

    except json.JSONDecodeError:

        pass


    # --------------------------------------------------------
    # 2. Markdown JSON block
    # --------------------------------------------------------

    match = re.search(
        r"```(?:json)?\s*(.*?)```",
        raw,
        re.DOTALL | re.IGNORECASE,
    )


    if match:

        try:

            parsed = json.loads(
                match.group(1).strip()
            )

            if isinstance(parsed, dict):

                return parsed

        except json.JSONDecodeError:

            pass


    # --------------------------------------------------------
    # 3. JSON-looking object inside surrounding text
    # --------------------------------------------------------

    start = raw.find("{")
    end = raw.rfind("}")


    if (
        start != -1
        and end > start
    ):

        try:

            parsed = json.loads(
                raw[start:end + 1]
            )

            if isinstance(parsed, dict):

                return parsed

        except json.JSONDecodeError:

            pass


    return None

In [81]:
def _fb_markdown(raw):
    """
    Recover JSON wrapped in a Markdown code block.
    """

    match = re.search(
        r"```(?:json)?\s*(\{.*?\})\s*```",
        raw,
        re.DOTALL,
    )


    if match:

        try:

            return json.loads(
                match.group(1)
            )

        except json.JSONDecodeError:

            return None


    return None

In [82]:
def _fb_literal_newline(raw):
    """
    Escape literal newline, carriage-return and tab characters
    that occur inside JSON strings.
    """

    text = raw.strip()


    if not text.startswith("{"):

        start = text.find("{")

        if start == -1:

            return None

        text = text[start:]


    depth = 0
    in_string = False
    escape = False
    end_index = -1


    for index, char in enumerate(text):

        if escape:

            escape = False
            continue


        if char == "\\":

            escape = True
            continue


        if char == '"':

            in_string = not in_string
            continue


        if in_string:

            continue


        if char == "{":

            depth += 1


        elif char == "}":

            depth -= 1

            if depth == 0:

                end_index = index
                break


    if end_index == -1:

        return None


    candidate = text[
        :end_index + 1
    ]


    result_chars = []

    in_string = False
    escape = False


    for char in candidate:

        if escape:

            result_chars.append(
                char
            )

            escape = False
            continue


        if char == "\\":

            result_chars.append(
                char
            )

            escape = True
            continue


        if char == '"':

            in_string = not in_string

            result_chars.append(
                char
            )

            continue


        if (
            in_string
            and char in "\n\r\t"
        ):

            if char == "\n":

                result_chars.append(
                    "\\n"
                )

            elif char == "\r":

                result_chars.append(
                    "\\r"
                )

            elif char == "\t":

                result_chars.append(
                    "\\t"
                )


        else:

            result_chars.append(
                char
            )


    fixed = "".join(
        result_chars
    )


    try:

        return json.loads(
            fixed
        )

    except json.JSONDecodeError:

        return None

In [83]:
def _fb_concatenated(raw):
    """
    Recover responses where reasoning was accidentally split
    into multiple adjacent strings.
    """

    text = raw.strip()


    if not text.startswith("{"):

        start = text.find("{")

        if start == -1:

            return None

        text = text[start:]


    reasoning_match = re.search(
        r'"reasoning"\s*:\s*"([^"]*)"',
        text,
        re.DOTALL,
    )


    answer_match = re.search(
        r'"answer"\s*:\s*(?:"([^"]*)"|([-+]?\d+(?:\.\d+)?)|(true|false|null))',
        text,
    )


    confidence_match = re.search(
        r'"confidence"\s*:\s*(\d+(?:\.\d+)?)',
        text,
    )


    if reasoning_match is None:

        return None


    reasoning = reasoning_match.group(
        1
    )


    end_of_reasoning = reasoning_match.end()


    if answer_match:

        start_of_answer = answer_match.start()

        middle = text[
            end_of_reasoning:start_of_answer
        ]


    else:

        middle = text[
            end_of_reasoning:
        ]


    orphan_strings = re.findall(
        r'"([^"]*)"',
        middle,
    )


    extra = " ".join(
        orphan_strings
    )


    if extra:

        reasoning = (
            f"{reasoning} {extra}"
            .strip()
        )


    if answer_match:

        answer = (
            answer_match.group(1)
            or answer_match.group(2)
            or answer_match.group(3)
        )

    else:

        answer = None


    if confidence_match:

        try:

            confidence = float(
                confidence_match.group(1)
            )

        except ValueError:

            confidence = None

    else:

        confidence = None


    if answer is None:

        return None


    return {
        "reasoning": reasoning,
        "answer": answer,

        "confidence": (
            confidence
            if confidence is not None
            else 50
        ),
    }

In [84]:
def _fb_regex(raw):
    """
    Final fallback: recover reasoning, answer and confidence
    with regular expressions.
    """

    reasoning_match = re.search(
        r'"reasoning"\s*:\s*"((?:[^"\\]|\\.)*?)"',
        raw,
        re.DOTALL,
    )


    answer_match = re.search(
        r'"answer"\s*:\s*(?:"((?:[^"\\]|\\.)*?)"|([-+]?\d+(?:\.\d+)?))',
        raw,
        re.DOTALL,
    )


    confidence_match = re.search(
        r'"confidence"\s*:\s*(\d+(?:\.\d+)?)',
        raw,
    )


    if (
        reasoning_match is None
        or answer_match is None
    ):

        return None


    reasoning = reasoning_match.group(
        1
    )


    answer = (
        answer_match.group(1)
        or answer_match.group(2)
    )


    confidence = (
        float(
            confidence_match.group(1)
        )
        if confidence_match
        else 50
    )


    return {
        "reasoning": reasoning,
        "answer": answer,
        "confidence": confidence,
    }

In [85]:
def extract_json_from_response(raw):
    """
    Parse a generated response.

    Normal extraction is attempted first.
    If it fails, the historical variant recovery strategies
    are attempted in sequence.
    """

    if not raw or not isinstance(raw, str):

        return None


    try:

        parsed = basic_extract_json(
            raw
        )


        if (
            isinstance(parsed, dict)
            and "reasoning" in parsed
            and "answer" in parsed
        ):

            return parsed


    except Exception:

        pass


    fallback_strategies = [
        _fb_markdown,
        _fb_literal_newline,
        _fb_concatenated,
        _fb_regex,
    ]


    for strategy in fallback_strategies:

        try:

            parsed = strategy(
                raw
            )


            if (
                isinstance(parsed, dict)
                and "reasoning" in parsed
                and "answer" in parsed
            ):

                return parsed


        except Exception:

            continue


    return None

In [86]:
# Parser sanity checks

markdown_test = extract_json_from_response(
    """```json
{"reasoning": "test", "answer": "42", "confidence": 80}
```"""
)


newline_test = extract_json_from_response(
    """{
"reasoning": "Adım 1.
Adım 2.",
"answer": 12.5,
"confidence": 100
}"""
)


assert (
    markdown_test is not None
    and str(markdown_test.get("answer")) == "42"
)


assert (
    newline_test is not None
    and float(newline_test.get("answer")) == 12.5
)


print("✓ Parser sanity checks passed.")

✓ Parser sanity checks passed.


## 8. Model loading and generation

Models are loaded one at a time using 4-bit quantization.

The historical variant-generation settings are:

- 4-bit loading,
- NF4 quantization,
- float16 computation,
- double quantization,
- greedy decoding (`do_sample=False`),
- `max_new_tokens=1024`.

After each model finishes, it is removed from GPU memory.

In [87]:
def load_model_4bit(model_id):

    print(
        f"Loading model: {model_id}"
    )


    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )


    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
    )


    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
    )


    model.eval()


    if tokenizer.pad_token is None:

        tokenizer.pad_token = (
            tokenizer.eos_token
        )


    print(
        f"✓ Model loaded: {model_id}"
    )


    return model, tokenizer

In [88]:
def supports_system_role(tokenizer):

    try:

        tokenizer.apply_chat_template(
            [
                {
                    "role": "system",
                    "content": "test",
                },

                {
                    "role": "user",
                    "content": "test",
                },
            ],

            tokenize=False,
            add_generation_prompt=True,
        )


        return True


    except Exception:

        return False

In [89]:
def generate_raw_response(
    model,
    tokenizer,
    context,
    question,
    max_new_tokens,
):

    user_prompt = build_user_prompt(
        context,
        question,
    )


    if supports_system_role(
        tokenizer
    ):

        messages = [
            {
                "role": "system",
                "content": OPEN_SOURCE_SYSTEM_PROMPT,
            },

            {
                "role": "user",
                "content": user_prompt,
            },
        ]


    else:

        combined_prompt = (
            OPEN_SOURCE_SYSTEM_PROMPT
            + "\n\n"
            + user_prompt
        )


        messages = [
            {
                "role": "user",
                "content": combined_prompt,
            }
        ]


    input_text = (
        tokenizer
        .apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    )


    inputs = tokenizer(
        input_text,
        return_tensors="pt",
    ).to(
        model.device
    )


    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )


    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]


    raw_response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )


    return raw_response

In [90]:
def cleanup_model(
    model,
    tokenizer,
):

    del model
    del tokenizer

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    print(
        "✓ GPU memory cleared."
    )

## 9. Output and checkpoint structure

Each selected model receives separate:

- raw output,
- metadata,
- checkpoint.

Variant outputs are stored under:

```text
results/raw/variants/

In [91]:
RAW_OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "raw"
    / "variants"
)


CHECKPOINT_DIR = (
    RAW_OUTPUT_DIR
    / "checkpoints"
)


def safe_name(value):

    return (
        str(value)
        .strip()
        .lower()
        .replace("/", "-")
        .replace(" ", "-")
    )


def get_run_name(model_label):

    return (
        f"{safe_name(DATASET_PROFILE)}__"
        f"{safe_name(model_label)}__"
        f"{safe_name(RUN_LABEL)}"
    )


def get_run_paths(model_label):

    run_name = get_run_name(
        model_label
    )


    return {
        "run_name": run_name,

        "checkpoint": (
            CHECKPOINT_DIR
            / f"{run_name}.json"
        ),

        "output": (
            RAW_OUTPUT_DIR
            / f"{run_name}.json"
        ),

        "metadata": (
            RAW_OUTPUT_DIR
            / f"{run_name}.metadata.json"
        ),
    }

In [92]:
print("Output plan:")


for model_label in SELECTED_MODELS:

    paths = get_run_paths(
        model_label
    )


    print(
        f"\n{model_label}"
    )


    print(
        f"  Checkpoint: "
        f"{paths['checkpoint']}"
    )


    print(
        f"  Output:     "
        f"{paths['output']}"
    )


    print(
        f"  Metadata:   "
        f"{paths['metadata']}"
    )

Output plan:

qwen2.5-7b-instruct
  Checkpoint: /content/drive/MyDrive/turkish-llm-reasoning/results/raw/variants/checkpoints/smoke__qwen2.5-7b-instruct__smoke_v1.json
  Output:     /content/drive/MyDrive/turkish-llm-reasoning/results/raw/variants/smoke__qwen2.5-7b-instruct__smoke_v1.json
  Metadata:   /content/drive/MyDrive/turkish-llm-reasoning/results/raw/variants/smoke__qwen2.5-7b-instruct__smoke_v1.metadata.json


## 10. Run metadata

Every run records enough information to identify the experimental condition
without relying on the notebook filename.

Metadata includes:

- dataset profile and hash,
- prompt hash,
- model identifier,
- quantization configuration,
- decoding configuration,
- record counts.

In [93]:
def utc_now():

    return (
        datetime
        .now(timezone.utc)
        .isoformat()
    )


def build_run_metadata(
    model_label,
):

    model_config = MODEL_REGISTRY[
        model_label
    ]


    return {
        "run_name": (
            get_run_name(
                model_label
            )
        ),

        "experiment_type": (
            "open_models_variants"
        ),

        "dataset_profile": (
            DATASET_PROFILE
        ),

        "dataset_path": str(
            VARIANT_DATA_PATH
        ),

        "dataset_sha256": (
            DATASET_SHA256
        ),

        "raw_dataset_rows": (
            len(raw_variants_df)
        ),

        "usable_dataset_rows": (
            len(variants_df)
        ),

        "unique_base_questions": (
            int(
                variants_df[
                    "base_question_id"
                ].nunique()
            )
        ),

        "variant_types": (
            int(
                variants_df[
                    "variant_type"
                ].nunique()
            )
        ),

        "prompt_sha256": (
            PROMPT_SHA256
        ),

        "parser": (
            "variant_fallback_v1"
        ),

        "model_label": (
            model_label
        ),

        "model_id": (
            model_config[
                "model_id"
            ]
        ),

        "historical_model": (
            model_config[
                "historical"
            ]
        ),

        "quantization": (
            "4bit_nf4"
        ),

        "do_sample": False,

        "max_new_tokens": (
            model_config[
                "max_new_tokens"
            ]
        ),
    }

## 11. Checkpoint helpers

Checkpoints allow long GPU experiments to resume after a runtime interruption.

Before resuming, the notebook verifies that the checkpoint belongs to the same:

- dataset,
- prompt,
- model,
- quantization configuration,
- decoding configuration.

This prevents incompatible runs from being combined.

In [94]:
def save_json_atomic(
    data,
    filepath,
):

    filepath = Path(
        filepath
    )


    filepath.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    temporary_file = (
        filepath
        .with_suffix(
            filepath.suffix + ".tmp"
        )
    )


    with open(
        temporary_file,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )


    os.replace(
        temporary_file,
        filepath,
    )

In [95]:
def load_checkpoint(
    checkpoint_file,
    current_metadata,
):

    checkpoint_file = Path(
        checkpoint_file
    )


    if not checkpoint_file.exists():

        return [], None


    with open(
        checkpoint_file,
        "r",
        encoding="utf-8",
    ) as file:

        checkpoint = json.load(
            file
        )


    saved_metadata = checkpoint[
        "metadata"
    ]

    records = checkpoint[
        "records"
    ]


    fields_to_validate = [
        "dataset_sha256",
        "prompt_sha256",
        "model_id",
        "quantization",
        "do_sample",
        "max_new_tokens",
    ]


    for field in fields_to_validate:

        if (
            saved_metadata.get(field)
            != current_metadata.get(field)
        ):

            raise ValueError(
                "Checkpoint configuration mismatch "
                f"for '{field}'."
            )


    print(
        "✓ Existing checkpoint loaded: "
        f"{len(records)} records"
    )


    return (
        records,
        saved_metadata.get(
            "created_at"
        ),
    )

## 12. Generate one model over the variant benchmark

Each model is processed independently.

For every variant:

1. the raw response is generated,
2. the raw response is preserved,
3. JSON parsing is attempted,
4. parsed fields and any errors are stored.

Parsing errors do not delete the raw response.

This allows later recovery without repeating expensive GPU generation.

In [96]:
def run_model_generation(
    model_label,
):

    model_config = MODEL_REGISTRY[
        model_label
    ]

    model_id = model_config[
        "model_id"
    ]

    max_new_tokens = model_config[
        "max_new_tokens"
    ]


    paths = get_run_paths(
        model_label
    )

    metadata = build_run_metadata(
        model_label
    )


    records, created_at = load_checkpoint(
        paths["checkpoint"],
        metadata,
    )


    if created_at is None:

        created_at = utc_now()


    processed_ids = {
        str(
            record["variant_id"]
        )

        for record in records
    }


    remaining_df = variants_df[
        ~variants_df["variant_id"]
        .astype(str)
        .isin(
            processed_ids
        )
    ]


    print(
        f"\n{'=' * 60}"
    )

    print(
        f"MODEL: "
        f"{model_config['display_name']}"
    )

    print(
        f"{'=' * 60}"
    )

    print(
        f"Already processed: "
        f"{len(processed_ids)}"
    )

    print(
        f"Remaining: "
        f"{len(remaining_df)}"
    )


    if len(remaining_df) == 0:

        print(
            "✓ No new variants need generation."
        )

        return pd.DataFrame(
            records
        )


    model = None
    tokenizer = None


    try:

        model, tokenizer = load_model_4bit(
            model_id
        )


        for _, row in tqdm(
            remaining_df.iterrows(),
            total=len(remaining_df),
            desc=model_label,
        ):

            raw_response = None


            try:

                raw_response = generate_raw_response(
                    model=model,
                    tokenizer=tokenizer,
                    context=str(
                        row["context"]
                    ),
                    question=str(
                        row["question"]
                    ),
                    max_new_tokens=max_new_tokens,
                )


                parsed = extract_json_from_response(
                    raw_response
                )


                if parsed is None:

                    reasoning = None
                    answer = None
                    confidence = None
                    error = "JSON_PARSE_ERROR"


                else:

                    reasoning = parsed.get(
                        "reasoning"
                    )

                    answer = parsed.get(
                        "answer"
                    )

                    confidence = parsed.get(
                        "confidence"
                    )

                    error = None


            except Exception as exc:

                reasoning = None
                answer = None
                confidence = None

                error = (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                )


            record = {
                "variant_id": str(
                    row["variant_id"]
                ),

                "base_question_id": str(
                    row["base_question_id"]
                ),

                "category": (
                    row["category"]
                ),

                "difficulty": (
                    row["difficulty"]
                ),

                "variant_type": (
                    row["variant_type"]
                ),

                "morphological_feature": (
                    row[
                        "morphological_feature"
                    ]
                ),

                "model_label": (
                    model_label
                ),

                "model_id": (
                    model_id
                ),

                "model_reasoning": (
                    reasoning
                ),

                "model_answer": (
                    answer
                ),

                "model_confidence": (
                    confidence
                ),

                "raw_response": (
                    raw_response
                ),

                "error": (
                    error
                ),
            }


            records.append(
                record
            )


            if (
                len(records)
                % CHECKPOINT_EVERY
                == 0
            ):

                checkpoint_payload = {
                    "metadata": {
                        **metadata,

                        "created_at": (
                            created_at
                        ),

                        "updated_at": (
                            utc_now()
                        ),

                        "status": (
                            "in_progress"
                        ),
                    },

                    "records": (
                        records
                    ),
                }


                save_json_atomic(
                    checkpoint_payload,
                    paths["checkpoint"],
                )


    finally:

        if (
            model is not None
            and tokenizer is not None
        ):

            cleanup_model(
                model,
                tokenizer,
            )


    final_metadata = {
        **metadata,

        "created_at": (
            created_at
        ),

        "completed_at": (
            utc_now()
        ),

        "status": (
            "completed"
        ),

        "record_count": (
            len(records)
        ),

        "successful_records": (
            sum(
                record.get("error") is None
                for record in records
            )
        ),

        "error_records": (
            sum(
                record.get("error") is not None
                for record in records
            )
        ),
    }


    final_payload = {
        "metadata": (
            final_metadata
        ),

        "records": (
            records
        ),
    }


    save_json_atomic(
        final_payload,
        paths["output"],
    )


    save_json_atomic(
        final_metadata,
        paths["metadata"],
    )


    save_json_atomic(
        final_payload,
        paths["checkpoint"],
    )


    print(
        f"\n✓ {model_label} completed."
    )

    print(
        f"Records: {len(records)}"
    )

    print(
        f"Output: {paths['output']}"
    )


    return pd.DataFrame(
        records
    )

## 13. Generation safety gate

This is the only section that starts model execution.

While `RUN_GENERATION = False`:

- no Hugging Face login is required,
- no model is downloaded,
- no GPU generation is performed.

If multiple models are selected,
`CONFIRM_MULTIPLE_MODELS = True` must also be explicitly enabled.

In [97]:
generation_results = {}


if RUN_GENERATION:

    if (
        len(SELECTED_MODELS) > 1
        and not CONFIRM_MULTIPLE_MODELS
    ):

        raise RuntimeError(
            "Multiple models are selected. "
            "Set CONFIRM_MULTIPLE_MODELS = True "
            "only if you intentionally want to run them."
        )


    login_to_huggingface_if_available()


    print(
        "Generation is ENABLED."
    )

    print(
        f"Variants per model: "
        f"{len(variants_df)}"
    )

    print(
        f"Selected models: "
        f"{len(SELECTED_MODELS)}"
    )


    for model_label in SELECTED_MODELS:

        generation_results[
            model_label
        ] = run_model_generation(
            model_label
        )


else:

    print(
        "Generation is disabled."
    )

    print(
        "No model was downloaded or executed."
    )

✓ Hugging Face authentication completed.
Generation is ENABLED.
Variants per model: 15
Selected models: 1

MODEL: Qwen 2.5 7B Instruct
Already processed: 0
Remaining: 15
Loading model: Qwen/Qwen2.5-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✓ Model loaded: Qwen/Qwen2.5-7B-Instruct


qwen2.5-7b-instruct:   0%|          | 0/15 [00:00<?, ?it/s]

✓ GPU memory cleared.

✓ qwen2.5-7b-instruct completed.
Records: 15
Output: /content/drive/MyDrive/turkish-llm-reasoning/results/raw/variants/smoke__qwen2.5-7b-instruct__smoke_v1.json


## 14. End of open-source variant generation notebook

This notebook intentionally ends after raw variant generation.

It does not perform:

- answer scoring,
- faithfulness evaluation,
- calibration analysis,
- base–variant comparison,
- model comparison,
- statistical analysis,
- figure generation.

Those stages are separated so that saved raw responses can be reprocessed
without repeating expensive GPU generation.

Generation notebooks:

- `01_closed_models_base.ipynb`
- `02_closed_models_variants.ipynb`
- `03_open_models_base.ipynb`
- `04_open_models_variants.ipynb`

Saved outputs from these notebooks will be evaluated in:

`05_evaluate_saved_outputs.ipynb`